# Img2Img + projection experiment

Use an input image to guide SDXL generation, then layer Joel's region projection on top during the back half of denoising. Three knobs to dial:

- **img2img strength** — how much to denoise the input image (lower = preserve more of the input).
- **projection_start_frac** — fraction of the *full* schedule at which Joel's projection kicks in.
- **projection alpha_end** — strength of the projection blend by the end of the trajectory.

**Subtlety to be aware of:** with img2img `strength<1.0`, only the *last `strength`-fraction* of the schedule actually runs. `projection_start_frac` is interpreted against the **full** schedule (matching the projector's existing logic), so if you want projection to ever fire, keep `projection_start_frac >= 1 - strength` (or set `strength` high enough that the trimmed schedule overlaps the projection window).

**Prereqs (one-time):** `pip install ipyfilechooser` (the file picker widget).

In [17]:
from __future__ import annotations

import importlib
import sys
from dataclasses import asdict, replace
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from ipyfilechooser import FileChooser

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import patch_dictionary_core
import render_config
import render_runtime
importlib.reload(patch_dictionary_core)
importlib.reload(render_config)
importlib.reload(render_runtime)

from render_config import GenerationConfig, ModelConfig, ProjectionConfig
from render_runtime import (
    DEVICE,
    PATCH_BANK_DTYPE,
    PIPELINE_DTYPE,
    build_runtime_patch_bank,
    load_pipeline,
    make_generator,
    make_projector_for_cfg,
    set_seed,
)

print(f"DEVICE={DEVICE}")

DEVICE=cuda


In [18]:
# Load the SDXL text2img pipeline. No LoRA by default — add fields to ModelConfig if you want one.
model_cfg = ModelConfig()
pipe = load_pipeline(model_cfg)
print("Loaded text2img pipeline")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loaded text2img pipeline


In [26]:
# Build the patch-bank source from a folder of your own images.
# First run encodes them; subsequent runs reuse the cached latents.
#
# `max_width` / `max_height` cap input dimensions before VAE encoding so the
# encode step doesn't OOM on large source photos. The downsize uses LANCZOS
# resampling (set in source_latents.py:72), so detail is well preserved at
# the cap. SDXL latents are 1/8 the input dimensions, so a 1536-px cap still
# gives a 192x192 latent grid per image — plenty for the patch bank. Bump to
# 2048 if you want richer per-image latents and have GPU headroom.
from source_latents import prepare_latents_from_images

SOURCE_IMAGES = Path("/home/nathan/Dropbox/Work/Experiments/070_Collage/01_InputSets/Wood")   # <-- change this
LATENTS_DIR = ROOT / "user-latents" / SOURCE_IMAGES.name
MAX_DIM = 1536   # cap on the long edge before VAE encoding; reduce to 1024 if you still OOM

if not LATENTS_DIR.exists() or not any(LATENTS_DIR.glob("*.npz")):
    LATENTS_DIR.mkdir(parents=True, exist_ok=True)
    prepare_latents_from_images(
        input_dir=SOURCE_IMAGES,
        output_dir=LATENTS_DIR,
        vae=pipe.vae,
        device=DEVICE,
        max_width=MAX_DIM,
        max_height=MAX_DIM,
        skip_existing=True,
    )
    print(f"Encoded latents into {LATENTS_DIR}")
else:
    print(f"Reusing existing latents in {LATENTS_DIR}")

Reusing existing latents in /home/nathan/AI/code/42_Joel_CollageNet/CollageNet/user-latents/Wood


In [27]:
# Projection + generation defaults. The sliders below override projection_start_frac and alpha_end per render.
patch_cfg = ProjectionConfig(
    latent_dir=LATENTS_DIR,
    region_method="felzenszwalb",
    patch_size=1,
    do_rotated=True,
    total_patches=50000,
    projection_start_frac=0.6,
    projection_end_frac=1.0,
    alpha_start=0.0,
    alpha_end=0.10,
    region_candidate_count=128,
    region_min_area=1,
    region_max_area=1200,
    felzenszwalb_scale=8.0,
    felzenszwalb_sigma=2.4,
    felzenszwalb_min_size=12,
)

gen_cfg = GenerationConfig(
    prompt="a portrait on a white background.",
    negative_prompt="blurry, low quality, deformed, extra limbs, text, watermark",
    height=1024,
    width=1024,
    num_inference_steps=30,
    guidance_scale=6.5,
    seed=1,
)

patch_bank = build_runtime_patch_bank(patch_cfg)
print(f"patch_bank.raw_patches.shape={tuple(patch_bank.raw_patches.shape)}")

patch_bank.raw_patches.shape=(50000, 4)


In [28]:
# Wrap the existing text2img pipeline as img2img. `from_pipe` shares all weights
# (text encoders, VAE, UNet) with the text2img pipe — no second download, no extra VRAM.
from diffusers import StableDiffusionXLImg2ImgPipeline

img2img_pipe = StableDiffusionXLImg2ImgPipeline.from_pipe(pipe)
print("Created img2img pipeline (shares weights with text2img pipe)")

Created img2img pipeline (shares weights with text2img pipe)


In [29]:
# --- Random-square region projector ----------------------------------------
# Tile the latent canvas with a random quadtree of squares. Sizes are powers
# of 2 (2, 4, 8, 16, ...) up to half the canvas, with the mix controlled by
# `min_size` (smallest tile allowed) and `split_prob` (probability of
# subdividing at each recursion level). Each leaf square becomes a region;
# the rest of the matching/projection machinery is inherited unchanged from
# LatentFelzenszwalbRegionProjector by overriding only the label-computing
# step.

import numpy as np
from patch_dictionary_core import (
    LatentFelzenszwalbRegionProjector,
    latent_labels_to_debug_image,
)


class LatentRandomSquareRegionProjector(LatentFelzenszwalbRegionProjector):
    def __init__(self, cfg, patch_bank, *, min_size: int = 2, split_prob: float = 0.65):
        super().__init__(cfg=cfg, patch_bank=patch_bank)
        self.random_square_min_size = max(1, int(min_size))
        self.random_square_split_prob = float(min(max(split_prob, 0.0), 1.0))

    def compute_felzenszwalb_labels(self, latents):
        """Override: produce labels via random quadtree subdivision."""
        _, _, h, w = latents.shape
        labels = np.full((h, w), -1, dtype=np.int32)
        next_label = [0]
        rng = self.rng
        min_size = self.random_square_min_size
        split_prob = self.random_square_split_prob

        def assign(y0, y1, x0, x1):
            ph = y1 - y0
            pw = x1 - x0
            if ph <= min_size or pw <= min_size or rng.random() > split_prob:
                labels[y0:y1, x0:x1] = next_label[0]
                next_label[0] += 1
                return
            ymid = y0 + ph // 2
            xmid = x0 + pw // 2
            assign(y0, ymid, x0, xmid)
            assign(y0, ymid, xmid, x1)
            assign(ymid, y1, x0, xmid)
            assign(ymid, y1, xmid, x1)

        assign(0, h, 0, w)
        return labels, {
            "random_square_min_size_used": float(min_size),
            "random_square_split_prob_used": float(split_prob),
            "num_random_squares": int(next_label[0]),
        }

    def maybe_save_debug_labels(self, labels, step_index, timestep):
        if self.debug_output_dir is None:
            return
        every_n = max(1, int(getattr(self.cfg, "debug_every_n_projections", 1)))
        if len(self.projection_events) % every_n != 0:
            return
        debug_image = latent_labels_to_debug_image(labels, seed=int(step_index))
        debug_path = self.debug_output_dir / f"random_square_step_{step_index:03d}_t{int(timestep):04d}.png"
        debug_image = debug_image.resize(
            (labels.shape[1] * 4, labels.shape[0] * 4),
            Image.Resampling.NEAREST,
        )
        debug_image.save(debug_path)


print("Defined LatentRandomSquareRegionProjector")

Defined LatentRandomSquareRegionProjector


In [30]:
# --- UI: path input, sliders, render button --------------------------------
from datetime import datetime
from PIL import ImageDraw, ImageFont

OUTPUT_DIR_EXP = ROOT / 'outputs' / 'img2img_experiment'
OUTPUT_DIR_EXP.mkdir(parents=True, exist_ok=True)

image_path_input = widgets.Text(
    value='', placeholder='paste an absolute path to an image (~ allowed)',
    description='image path',
    style={'description_width': '180px'}, layout={'width': '700px'},
)

prompt_text = widgets.Text(
    value=gen_cfg.prompt, description='prompt',
    style={'description_width': '180px'}, layout={'width': '700px'},
)

_slider_layout = {'width': '500px'}
_label_style = {'description_width': '180px'}


def _desc(text: str) -> widgets.HTML:
    """Short caption for the control above; sits indented under the slider label."""
    return widgets.HTML(
        f'<div style="margin-left: 184px; margin-bottom: 10px; '
        f'color: #888; font-size: 0.85em; max-width: 600px;">{text}</div>'
    )


# --- Step counts ---
skip_slider = widgets.IntSlider(
    value=9, min=0, max=29, step=1,
    description='skip_steps (img2img)',
    style=_label_style, layout=_slider_layout,
)
text2img_slider = widgets.IntSlider(
    value=12, min=1, max=40, step=1,
    description='text2img_steps',
    style=_label_style, layout=_slider_layout,
)
projection_slider = widgets.IntSlider(
    value=9, min=0, max=40, step=1,
    description='projection_steps',
    style=_label_style, layout=_slider_layout,
)
alpha_end_slider = widgets.FloatSlider(
    value=0.10, min=0.0, max=1.0, step=0.05,
    description='projection alpha_end',
    style=_label_style, layout=_slider_layout,
)

# --- Patch bank shape (changing region_method/patch_size triggers a rebuild) ---
region_method_dropdown = widgets.Dropdown(
    options=['felzenszwalb', 'threshold', 'square', 'random_square'],
    value=patch_cfg.region_method,
    description='region_method',
    style=_label_style, layout={'width': '500px'},
)
# patch_size only takes legal factors of the latent grid (1024 image -> 128
# latent cells). Pickable values ensure no "must be divisible by N" errors.
patch_size_slider = widgets.SelectionSlider(
    options=[1, 2, 4, 8, 16],
    value=patch_cfg.patch_size if patch_cfg.patch_size in (1, 2, 4, 8, 16) else 1,
    description='patch_size (square only)',
    style=_label_style, layout=_slider_layout,
)

# --- Felzenszwalb shape knobs (only matter when region_method=felzenszwalb) ---
felz_scale_slider = widgets.FloatSlider(
    value=patch_cfg.felzenszwalb_scale, min=1.0, max=100.0, step=1.0,
    description='felz_scale',
    style=_label_style, layout=_slider_layout,
)
felz_sigma_slider = widgets.FloatSlider(
    value=patch_cfg.felzenszwalb_sigma, min=0.1, max=4.0, step=0.1,
    description='felz_sigma',
    style=_label_style, layout=_slider_layout,
)
felz_min_size_slider = widgets.IntSlider(
    value=patch_cfg.felzenszwalb_min_size, min=1, max=64, step=1,
    description='felz_min_size',
    style=_label_style, layout=_slider_layout,
)

# --- Random-square knobs (only matter when region_method=random_square) ---
rs_min_size_slider = widgets.IntSlider(
    value=2, min=1, max=16, step=1,
    description='rs_min_size',
    style=_label_style, layout=_slider_layout,
)
rs_split_prob_slider = widgets.FloatSlider(
    value=0.65, min=0.0, max=1.0, step=0.05,
    description='rs_split_prob',
    style=_label_style, layout=_slider_layout,
)

seed_int = widgets.IntText(
    value=1, description='seed',
    style=_label_style, layout={'width': '300px'},
)

derived_label = widgets.HTML()


def _refresh_derived(*_):
    skip = skip_slider.value
    free = text2img_slider.value
    proj = projection_slider.value
    total = skip + free + proj
    if total <= 0:
        derived_label.value = "<i>need at least one step</i>"
        return
    strength = (free + proj) / total
    proj_start = (skip + free) / total
    derived_label.value = (
        f"<code>num_inference_steps={total} &nbsp; "
        f"strength={strength:.3f} &nbsp; "
        f"projection_start_frac={proj_start:.3f}</code>"
    )


for w in (skip_slider, text2img_slider, projection_slider):
    w.observe(_refresh_derived, names='value')
_refresh_derived()

render_btn = widgets.Button(description='Render', button_style='primary')
output_area = widgets.Output()

display(
    image_path_input, prompt_text,
    widgets.HTML('<b>Step counts</b>'),
    skip_slider,
    _desc("Skip the first N steps of the full schedule. img2img starts here — "
          "the noise level matches the timestep at that point. Higher = preserves more of input."),
    text2img_slider,
    _desc("Free denoising steps with prompt only — projection is off here. "
          "More = the prompt gets more influence over structure."),
    projection_slider,
    _desc("Trailing steps where Joel's projection fires, pulling the latent "
          "toward the source patch bank. More = stronger collage effect."),
    alpha_end_slider,
    _desc("Projection blend at the final step (linear ramp from 0). "
          "0.05–0.15 is subtle, 0.3+ starts to dominate, 0.5+ usually noisy."),
    derived_label,
    widgets.HTML('<b>Patch bank shape</b> (changes here trigger a rebuild on the next render)'),
    region_method_dropdown,
    _desc("How the live latent is segmented for matching. "
          "<b>felzenszwalb</b> = irregular merged regions; "
          "<b>threshold</b> = similarity-based regions; "
          "<b>square</b> = fixed grid (uses patch_size); "
          "<b>random_square</b> = quadtree of mixed-size squares (uses rs_min_size, rs_split_prob)."),
    patch_size_slider,
    _desc("Latent cells per patch. Only used by region_method=square. "
          "Restricted to factors of the latent grid (1, 2, 4, 8, 16) to avoid divisibility errors."),
    widgets.HTML('<b>Felzenszwalb knobs</b> (only matter when region_method=felzenszwalb)'),
    felz_scale_slider,
    _desc("Region-size pressure. Higher tends to merge into bigger regions, "
          "but the relationship to min_size is nonlinear."),
    felz_sigma_slider,
    _desc("Gaussian smoothing applied before segmentation. "
          "Higher = softer/cleaner region boundaries."),
    felz_min_size_slider,
    _desc("Minimum region size in latent cells. "
          "Low (1–6) = many small shards; high (20+) = few big blocks."),
    widgets.HTML('<b>Random-square knobs</b> (only matter when region_method=random_square)'),
    rs_min_size_slider,
    _desc("Smallest tile size (in latent cells) the quadtree is allowed to produce. "
          "Lower = chance of very tiny tiles; higher = floor on tile size."),
    rs_split_prob_slider,
    _desc("Probability of subdividing at each recursion. "
          "0.0 = one giant tile, 1.0 = always subdivide to min_size, "
          "~0.6 gives a nice mix of large and small."),
    seed_int, render_btn, output_area,
)


# --- Helpers (shared by single-render handler and the sweep cell below) -----

def _snap8(v: int) -> int:
    return max(8, v - (v % 8))


def _resolve_image_path(raw: str) -> Path | None:
    raw = (raw or '').strip().strip('"').strip("'")
    if not raw:
        return None
    return Path(raw).expanduser()


def _side_by_side(left: Image.Image, right: Image.Image) -> Image.Image:
    """Compose two images side-by-side; left is rescaled to match right's size."""
    if left.size != right.size:
        left = left.resize(right.size, Image.Resampling.LANCZOS)
    w, h = right.size
    combined = Image.new('RGB', (w * 2, h), (255, 255, 255))
    combined.paste(left, (0, 0))
    combined.paste(right, (w, 0))
    return combined


def _load_font(size: int):
    """Try a few common monospace font paths; fall back to PIL default."""
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
        "/Library/Fonts/Menlo.ttc",
        "C:/Windows/Fonts/consola.ttf",
    ):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()


def _label_image(img: Image.Image, text: str) -> Image.Image:
    """Draw multi-line text in a small dark box at top-left; returns a copy."""
    base = img.convert('RGBA')
    overlay = Image.new('RGBA', base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    font = _load_font(20)
    pad, margin = 10, 12
    bbox = draw.multiline_textbbox((margin, margin), text, font=font, spacing=4)
    box = (bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad)
    draw.rectangle(box, fill=(0, 0, 0, 200))
    draw.multiline_text((margin, margin), text, fill=(255, 255, 255, 255),
                        font=font, spacing=4)
    return Image.alpha_composite(base, overlay).convert('RGB')


# Track the params the current `patch_bank` was built with so we know when to rebuild.
_current_bank_params = {
    'region_method': patch_cfg.region_method,
    'patch_size': patch_cfg.patch_size,
    'do_rotated': patch_cfg.do_rotated,
    'total_patches': patch_cfg.total_patches,
}


def _ensure_bank(region_method: str, patch_size: int):
    global patch_bank, _current_bank_params
    # Region methods (everything except 'square') always use patch_size=1 in
    # the bank — patches are individual latent cells, regions are made by
    # combining those cells. Forcing patch_size=1 here avoids needless rebuilds
    # when toggling between region methods with stale slider values.
    effective_patch_size = patch_size if region_method == 'square' else 1
    requested = {
        'region_method': region_method,
        'patch_size': effective_patch_size,
        'do_rotated': patch_cfg.do_rotated,
        'total_patches': patch_cfg.total_patches,
    }
    if requested == _current_bank_params:
        return
    rebuild_cfg = replace(
        patch_cfg,
        region_method=region_method,
        patch_size=effective_patch_size,
    )
    print(f"Rebuilding patch bank: {requested}  (this takes a few seconds...)")
    patch_bank = build_runtime_patch_bank(rebuild_cfg)
    _current_bank_params = requested


def _render_one(*, image_path, prompt, skip, free, proj, alpha_end,
                region_method, patch_size, felz_scale, felz_sigma, felz_min_size,
                rs_min_size, rs_split_prob,
                seed):
    """Render one combo. Returns (original_input, diffusion, sharp_or_None, params_dict)."""
    total = skip + free + proj
    if total <= 0:
        raise ValueError("Need at least one step.")

    strength = min(max((free + proj) / total, 1e-3), 1.0)
    proj_start_frac = min(max((skip + free) / total, 0.0), 1.0)

    _ensure_bank(region_method, patch_size)

    original_input = Image.open(image_path).convert("RGB")
    target_w = _snap8(gen_cfg.width)
    target_h = _snap8(gen_cfg.height)
    init_image = original_input.resize((target_w, target_h), Image.Resampling.LANCZOS)

    live_patch_cfg = replace(
        patch_cfg,
        # random_square is dispatched manually below; the cfg's projector_mode
        # mapping doesn't know about it. Set region_method to felzenszwalb in
        # the cfg so any internal lookups don't trip.
        region_method=region_method if region_method != 'random_square' else 'felzenszwalb',
        patch_size=patch_size if region_method == 'square' else 1,
        projection_start_frac=proj_start_frac,
        alpha_end=alpha_end,
        felzenszwalb_scale=felz_scale,
        felzenszwalb_sigma=felz_sigma,
        felzenszwalb_min_size=felz_min_size,
    )

    if region_method == 'random_square':
        projector = LatentRandomSquareRegionProjector(
            cfg=live_patch_cfg, patch_bank=patch_bank,
            min_size=rs_min_size, split_prob=rs_split_prob,
        )
    else:
        projector = make_projector_for_cfg(live_patch_cfg, patch_bank)

    set_seed(seed)
    result = img2img_pipe(
        prompt=prompt,
        negative_prompt=gen_cfg.negative_prompt,
        image=init_image,
        strength=strength,
        num_inference_steps=total,
        guidance_scale=gen_cfg.guidance_scale,
        generator=make_generator(seed),
        callback_on_step_end=projector,
        callback_on_step_end_tensor_inputs=["latents"],
    )
    diffusion_image = result.images[0]

    sharp_image = None
    if (
        getattr(projector, 'final_region_assignments', None) is not None
        and getattr(projector, 'final_assignment_grid_shape', None) is not None
    ):
        sharp_image = patch_dictionary_core.render_pixel_collage_from_region_assignments(
            patch_bank=patch_bank,
            region_assignments=projector.final_region_assignments,
            latent_canvas_shape=projector.final_assignment_grid_shape,
            pixel_render_scale=1,
        )
    elif (
        getattr(projector, 'final_selected_patch_indices', None) is not None
        and getattr(projector, 'final_assignment_grid_shape', None) is not None
    ):
        sharp_image = patch_dictionary_core.render_pixel_collage_from_assignments(
            patch_bank=patch_bank,
            selected_patch_indices=projector.final_selected_patch_indices.numpy()[0],
            grid_shape=projector.final_assignment_grid_shape,
            patch_size=live_patch_cfg.patch_size,
            pixel_render_scale=1,
        )

    params = {
        'prompt': prompt, 'seed': seed,
        'skip': skip, 'free': free, 'proj': proj,
        'strength': strength, 'proj_start_frac': proj_start_frac,
        'alpha_end': alpha_end,
        'region_method': region_method, 'patch_size': patch_size,
        'felz_scale': felz_scale, 'felz_sigma': felz_sigma, 'felz_min_size': felz_min_size,
        'rs_min_size': rs_min_size, 'rs_split_prob': rs_split_prob,
    }
    return original_input, diffusion_image, sharp_image, params


def on_render_click(_btn):
    with output_area:
        output_area.clear_output()

        image_path = _resolve_image_path(image_path_input.value)
        if image_path is None:
            print("Paste an image path first.")
            return
        if not image_path.is_file():
            print(f"Not a file: {image_path}")
            return

        try:
            original_input, diffusion_image, sharp_image, params = _render_one(
                image_path=image_path,
                prompt=prompt_text.value,
                skip=skip_slider.value,
                free=text2img_slider.value,
                proj=projection_slider.value,
                alpha_end=alpha_end_slider.value,
                region_method=region_method_dropdown.value,
                patch_size=patch_size_slider.value,
                felz_scale=felz_scale_slider.value,
                felz_sigma=felz_sigma_slider.value,
                felz_min_size=felz_min_size_slider.value,
                rs_min_size=rs_min_size_slider.value,
                rs_split_prob=rs_split_prob_slider.value,
                seed=seed_int.value,
            )
        except ValueError as e:
            print(str(e))
            return

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        run_label = f"{timestamp}_seed{seed_int.value:04d}_{image_path.stem}"
        run_dir = OUTPUT_DIR_EXP / run_label
        run_dir.mkdir(parents=True, exist_ok=True)

        diffusion_image.save(run_dir / 'diffusion.png')
        if sharp_image is not None:
            sidebyside_img = _side_by_side(original_input, sharp_image)
            sharp_image.save(run_dir / 'sharp.png')
            sidebyside_img.save(run_dir / 'sidebyside.png')
        else:
            sidebyside_img = _side_by_side(original_input, diffusion_image)
            sidebyside_img.save(run_dir / 'sidebyside.png')

        (run_dir / 'params.txt').write_text(
            f"prompt: {params['prompt']}\n"
            f"negative_prompt: {gen_cfg.negative_prompt}\n"
            f"input: {image_path}\n"
            f"seed: {params['seed']}\n"
            f"skip_steps: {params['skip']}\n"
            f"text2img_steps: {params['free']}\n"
            f"projection_steps: {params['proj']}\n"
            f"num_inference_steps: {params['skip']+params['free']+params['proj']}\n"
            f"strength: {params['strength']:.4f}\n"
            f"projection_start_frac: {params['proj_start_frac']:.4f}\n"
            f"alpha_end: {params['alpha_end']}\n"
            f"region_method: {params['region_method']}\n"
            f"patch_size: {params['patch_size']}\n"
            f"felz_scale: {params['felz_scale']}\n"
            f"felz_sigma: {params['felz_sigma']}\n"
            f"felz_min_size: {params['felz_min_size']}\n"
            f"rs_min_size: {params['rs_min_size']}\n"
            f"rs_split_prob: {params['rs_split_prob']}\n"
        )

        print(
            f"steps: skip={params['skip']} text2img={params['free']} projection={params['proj']}  "
            f"(total={params['skip']+params['free']+params['proj']}, "
            f"strength={params['strength']:.3f}, "
            f"projection_start_frac={params['proj_start_frac']:.3f})  "
            f"alpha_end={params['alpha_end']:.2f}  seed={params['seed']}  "
            f"input={image_path.name}"
        )
        if params['region_method'] == 'random_square':
            print(
                f"shape: region_method=random_square  "
                f"rs=(min_size={params['rs_min_size']}, split_prob={params['rs_split_prob']:.2f})"
            )
        elif params['region_method'] == 'felzenszwalb':
            print(
                f"shape: region_method=felzenszwalb  "
                f"felz=(scale={params['felz_scale']:.1f}, sigma={params['felz_sigma']:.2f}, "
                f"min_size={params['felz_min_size']})"
            )
        else:
            print(
                f"shape: region_method={params['region_method']}  "
                f"patch_size={params['patch_size']}"
            )
        print(f"saved -> {run_dir}")

        if sharp_image is not None:
            print("\nInput | Sharp pixel collage:")
            display(sidebyside_img)
            print("\nDiffusion output (VAE pass):")
            display(diffusion_image)
        else:
            print(
                "\n(No sharp collage rendered — projection never fired. "
                "Showing input | diffusion comparison instead.)"
            )
            print("\nInput | Diffusion output:")
            display(sidebyside_img)


render_btn.on_click(on_render_click)

Text(value='', description='image path', layout=Layout(width='700px'), placeholder='paste an absolute path to …

Text(value='a portrait on a white background.', description='prompt', layout=Layout(width='700px'), style=Text…

HTML(value='<b>Step counts</b>')

IntSlider(value=9, description='skip_steps (img2img)', layout=Layout(width='500px'), max=29, style=SliderStyle…

HTML(value='<div style="margin-left: 184px; margin-bottom: 10px; color: #888; font-size: 0.85em; max-width: 60…

...

FloatSlider(value=0.65, description='rs_split_prob', layout=Layout(width='500px'), max=1.0, step=0.05, style=S…

HTML(value='<div style="margin-left: 184px; margin-bottom: 10px; color: #888; font-size: 0.85em; max-width: 60…

IntText(value=1, description='seed', layout=Layout(width='300px'), style=DescriptionStyle(description_width='1…

Button(button_style='primary', description='Render', style=ButtonStyle())

Output()

In [24]:
# --- Sweep button: vary one parameter at a time across low/mid/high ---------
# Walks several knobs through low/mid/high values one at a time, holding all
# the other knobs at the current slider settings. Saves three versions of
# each variant (diffusion / sharp / labeled) into a single sweep folder, plus
# one input|sharp side-by-side for the baseline.
#
# Filename scheme (so files of the same kind group together when sorted):
#   diffusion__<idx>_<param>_<val>.png    — VAE pass before collage refine
#   sharp__<idx>_<param>_<val>.png        — sharp pixel collage
#   labeled__<idx>_<param>_<val>.png      — sharp + settings overlay
#   sidebyside__00_baseline.png           — only one of these

sweep_btn = widgets.Button(description='Generate sweep', button_style='warning')
sweep_output = widgets.Output()
display(sweep_btn, sweep_output)


def _format_val_for_filename(val) -> str:
    if isinstance(val, float):
        return f"{val:.2f}".replace('.', '_')
    return str(val).replace(' ', '_').replace('/', '_')


def _format_label_text(p: dict, highlight: str | None = None) -> str:
    """Multi-line text drawn onto the labeled images."""
    lines = []
    if highlight:
        lines.append(f">>> {highlight} <<<")
    lines.append(f"seed={p['seed']}")
    lines.append(
        f"skip={p['skip']}  free={p['free']}  proj={p['proj']}  "
        f"total={p['skip'] + p['free'] + p['proj']}"
    )
    lines.append(f"strength={p['strength']:.3f}  proj_start={p['proj_start_frac']:.3f}")
    lines.append(f"alpha_end={p['alpha_end']:.2f}")
    lines.append(f"region={p['region_method']}  patch={p['patch_size']}")
    if p['region_method'] == 'random_square':
        lines.append(
            f"rs: min_size={p.get('rs_min_size', '?')}  "
            f"split_prob={p.get('rs_split_prob', 0.0):.2f}"
        )
    elif p['region_method'] == 'felzenszwalb':
        lines.append(
            f"felz: scale={p['felz_scale']:.1f}  sigma={p['felz_sigma']:.2f}  "
            f"min={p['felz_min_size']}"
        )
    prompt_trunc = p['prompt'] if len(p['prompt']) <= 60 else p['prompt'][:57] + '...'
    lines.append(f"prompt: {prompt_trunc}")
    return '\n'.join(lines)


def on_sweep_click(_btn):
    with sweep_output:
        sweep_output.clear_output()

        image_path = _resolve_image_path(image_path_input.value)
        if image_path is None:
            print("Paste an image path first.")
            return
        if not image_path.is_file():
            print(f"Not a file: {image_path}")
            return

        baseline = dict(
            image_path=image_path,
            prompt=prompt_text.value,
            skip=skip_slider.value,
            free=text2img_slider.value,
            proj=projection_slider.value,
            alpha_end=alpha_end_slider.value,
            region_method=region_method_dropdown.value,
            patch_size=patch_size_slider.value,
            felz_scale=felz_scale_slider.value,
            felz_sigma=felz_sigma_slider.value,
            felz_min_size=felz_min_size_slider.value,
            rs_min_size=rs_min_size_slider.value,
            rs_split_prob=rs_split_prob_slider.value,
            seed=seed_int.value,
        )

        # Sweep schedule: vary one knob at a time across low / mid / high.
        sweeps = [
            ('alpha_end', [0.05, 0.20, 0.50]),
            ('skip', [0, 9, 20]),
            ('proj', [3, 12, 24]),
            ('region_method', ['felzenszwalb', 'threshold', 'square', 'random_square']),
            ('felz_min_size', [2, 12, 32]),
            ('rs_split_prob', [0.30, 0.60, 0.90]),
        ]

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        sweep_dir = OUTPUT_DIR_EXP / f"sweep_{timestamp}_{image_path.stem}"
        sweep_dir.mkdir(parents=True, exist_ok=True)

        total_renders = 1 + sum(len(values) for _, values in sweeps)
        print(f"Sweep: {total_renders} renders -> {sweep_dir}")

        original_input = Image.open(baseline['image_path']).convert('RGB')

        # 1. Baseline.
        print(f"\n[1/{total_renders}] baseline (current slider values)...")
        try:
            _, baseline_diffusion, baseline_sharp, params = _render_one(**baseline)
        except Exception as e:
            print(f"  ERROR: {e}")
            return

        baseline_diffusion.save(sweep_dir / 'diffusion__00_baseline.png')
        baseline_sidebyside = None
        if baseline_sharp is not None:
            baseline_sidebyside = _side_by_side(original_input, baseline_sharp)
            baseline_sidebyside.save(sweep_dir / 'sidebyside__00_baseline.png')
            baseline_sharp.save(sweep_dir / 'sharp__00_baseline.png')
            _label_image(
                baseline_sharp, _format_label_text(params, 'baseline')
            ).save(sweep_dir / 'labeled__00_baseline.png')
        else:
            print("  (baseline produced no sharp — labeled/sidebyside skipped)")

        # 2. Variants.
        idx = 1
        for sweep_idx, (var_name, values) in enumerate(sweeps, start=1):
            for val in values:
                idx += 1
                variant = dict(baseline)
                variant[var_name] = val

                stem = f"{sweep_idx:02d}_{var_name}_{_format_val_for_filename(val)}"
                print(f"[{idx}/{total_renders}] {stem}...")

                try:
                    _, diffusion, sharp, params = _render_one(**variant)
                except Exception as e:
                    print(f"  ERROR: {e}")
                    continue

                diffusion.save(sweep_dir / f"diffusion__{stem}.png")

                if sharp is None:
                    print("  (no sharp produced)")
                    continue

                sharp.save(sweep_dir / f"sharp__{stem}.png")
                _label_image(
                    sharp, _format_label_text(params, f"{var_name}={val}")
                ).save(sweep_dir / f"labeled__{stem}.png")

        (sweep_dir / 'sweep_info.txt').write_text(
            "baseline (current slider values when sweep was triggered):\n"
            + '\n'.join(f"  {k}: {v}" for k, v in baseline.items())
            + "\n\nsweeps (one knob varied at a time, others held at baseline):\n"
            + '\n'.join(f"  {name}: {values}" for name, values in sweeps)
            + "\n"
        )

        print(f"\n[done] saved {total_renders} renders to:\n  {sweep_dir}")
        if baseline_sidebyside is not None:
            print("\nBaseline (input | sharp):")
            display(baseline_sidebyside)


sweep_btn.on_click(on_sweep_click)

Button(button_style='warning', description='Generate sweep', style=ButtonStyle())

Output()

In [25]:
# --- Party mode: many random combos -----------------------------------------
# Generates N renders with each knob randomized inside a "interesting" range.
# Uses the input image and prompt from the UI; everything else is rolled fresh
# per render. Saves diffusion / sharp / labeled triplets and a CSV-ish log so
# you can recreate any specific render later by copying its values.
#
# Reuses _render_one, _label_image, _resolve_image_path, _side_by_side from
# the UI cell, and _format_label_text from the sweep cell — re-run those
# cells first if you've restarted the kernel.

import random as _random

party_count_slider = widgets.IntSlider(
    value=12, min=1, max=40, step=1,
    description='party count',
    style=_label_style, layout=_slider_layout,
)
party_region_dropdown = widgets.Dropdown(
    options=['randomize', 'felzenszwalb', 'threshold', 'square', 'random_square'],
    value='randomize',
    description='region_method',
    style=_label_style, layout={'width': '500px'},
)
party_btn = widgets.Button(description='Party mode 🎉', button_style='success')
party_output = widgets.Output()
display(
    party_count_slider,
    party_region_dropdown,
    _desc("Pin region_method to one value (avoids the patch-bank rebuild between "
          "variants and keeps you focused on one segmentation style), "
          "or 'randomize' to roll it fresh each render."),
    party_btn, party_output,
)


# Random ranges. Tweak these to make the party tamer or wilder.
_party_ranges = {
    'skip':         (0, 25),                  # int range
    'free':         (4, 25),                  # int range
    'proj':         (0, 25),                  # int range (0 = no projection)
    'alpha_end':    (0.05, 0.60),             # float range
    'patch_size':   (1, 4),                   # int range
    'felz_scale':   (4.0, 50.0),              # float range
    'felz_sigma':   (0.5, 3.5),               # float range
    'felz_min_size':(2, 32),                  # int range
    'rs_min_size':  (1, 8),                   # int range
    'rs_split_prob':(0.30, 0.90),             # float range
    'region_method':['felzenszwalb', 'threshold', 'square', 'random_square'],  # choices
}


def _roll_combo(rng, image_path, prompt, *, fixed_region_method: str | None = None) -> dict:
    region = (
        fixed_region_method
        if fixed_region_method is not None
        else rng.choice(_party_ranges['region_method'])
    )
    return dict(
        image_path=image_path,
        prompt=prompt,
        skip=rng.randint(*_party_ranges['skip']),
        free=rng.randint(*_party_ranges['free']),
        proj=rng.randint(*_party_ranges['proj']),
        alpha_end=round(rng.uniform(*_party_ranges['alpha_end']), 2),
        region_method=region,
        patch_size=rng.choice([1, 2, 4, 8]),  # legal grid factors only
        felz_scale=round(rng.uniform(*_party_ranges['felz_scale']), 1),
        felz_sigma=round(rng.uniform(*_party_ranges['felz_sigma']), 2),
        felz_min_size=rng.randint(*_party_ranges['felz_min_size']),
        rs_min_size=rng.randint(*_party_ranges['rs_min_size']),
        rs_split_prob=round(rng.uniform(*_party_ranges['rs_split_prob']), 2),
        seed=rng.randint(0, 2**31 - 1),
    )


def on_party_click(_btn):
    with party_output:
        party_output.clear_output()

        image_path = _resolve_image_path(image_path_input.value)
        if image_path is None:
            print("Paste an image path first.")
            return
        if not image_path.is_file():
            print(f"Not a file: {image_path}")
            return

        N = party_count_slider.value
        fixed_region = (
            None if party_region_dropdown.value == 'randomize'
            else party_region_dropdown.value
        )

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        region_tag = fixed_region if fixed_region else 'mixed'
        party_dir = OUTPUT_DIR_EXP / f"party_{timestamp}_{region_tag}_{image_path.stem}"
        party_dir.mkdir(parents=True, exist_ok=True)

        rng = _random.Random()
        if fixed_region:
            print(f"Party mode 🎉  {N} renders, region pinned to '{fixed_region}'")
        else:
            print(f"Party mode 🎉  {N} renders, region_method randomized per render")
        print(f"-> {party_dir}")

        log_lines = ["idx,seed,skip,free,proj,alpha_end,region_method,patch_size,felz_scale,felz_sigma,felz_min_size,rs_min_size,rs_split_prob"]

        for i in range(1, N + 1):
            combo = _roll_combo(
                rng, image_path, prompt_text.value,
                fixed_region_method=fixed_region,
            )
            stem = f"{i:03d}"
            print(
                f"[{i:03d}/{N}] region={combo['region_method']:13s} "
                f"skip={combo['skip']:2d} free={combo['free']:2d} proj={combo['proj']:2d}  "
                f"alpha={combo['alpha_end']:.2f}  seed={combo['seed']}"
            )

            try:
                _, diffusion, sharp, params = _render_one(**combo)
            except Exception as e:
                print(f"  ERROR: {e}")
                continue

            diffusion.save(party_dir / f"diffusion__{stem}.png")
            label_text = _format_label_text(params, f"party #{stem}")
            if sharp is not None:
                sharp.save(party_dir / f"sharp__{stem}.png")
                _label_image(sharp, label_text).save(party_dir / f"labeled__{stem}.png")
            else:
                # No projection fired → label the diffusion image so the
                # `labeled__` series stays unbroken when scrolling.
                _label_image(diffusion, label_text).save(party_dir / f"labeled__{stem}.png")

            log_lines.append(
                f"{stem},{combo['seed']},{combo['skip']},{combo['free']},{combo['proj']},"
                f"{combo['alpha_end']:.2f},{combo['region_method']},{combo['patch_size']},"
                f"{combo['felz_scale']:.1f},{combo['felz_sigma']:.2f},{combo['felz_min_size']},"
                f"{combo['rs_min_size']},{combo['rs_split_prob']:.2f}"
            )

        (party_dir / 'party_log.csv').write_text('\n'.join(log_lines) + '\n')

        print(f"\n[done] {N} renders + party_log.csv written to:\n  {party_dir}")


party_btn.on_click(on_party_click)

IntSlider(value=12, description='party count', layout=Layout(width='500px'), max=40, min=1, style=SliderStyle(…

Dropdown(description='region_method', layout=Layout(width='500px'), options=('randomize', 'felzenszwalb', 'thr…

HTML(value='<div style="margin-left: 184px; margin-bottom: 10px; color: #888; font-size: 0.85em; max-width: 60…

Button(button_style='success', description='Party mode 🎉', style=ButtonStyle())

Output()